## Synthetic spectra generation (H2-He sub-Neptune / K2-18b analogue)

This notebook generates the synthetic transmission-spectra grid used in the
sub-Neptune G-DAE workflow. The case is labelled as a K2-18b analogue: an
H2-He dominated atmosphere with trace gases sampled over the parameter space
listed below.

The first cells prepare the local MultiREx/TauREx inputs. The gas-opacity,
PHOENIX stellar, and CIA files are downloaded or refreshed by the setup calls
when they are not already present locally; these heavy directories are kept out
of the GitHub repository.

### Atmosphere (primary)
- Bulk composition: **H2-He** (CIA terms: **H2-H2** and **H2-He**).
- Pressure bounds are fixed to:
  - Base pressure: **10 bar** (implemented as `base_pressure=10e5` Pa = `1e6` Pa).
  - Top pressure: **1e-8 bar** (implemented as `top_pressure=1e-3` Pa).

### Temperature grid
The explored atmospheric temperatures are:
- **[250, 300, 350, 400, 450] K**

### Trace gases and explored log10(VMR) ranges
The following trace gases are included (log10 volume mixing ratios):
- **H2O**: from **-7 to -2**, with **absence allowed** (`include_absence=True`)
- **CO2**: from **-10 to -1**
- **CH4**: from **-8 to -1**, with **absence allowed**
- **NH3**: from **-7 to -2**, with **absence allowed**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import multirex as mrex

# Prepare local opacity and stellar-input folders used by the spectra generator.
mrex.Util.get_gases()
mrex.Util.get_stellar_phoenix()
mrex.Util.list_gases()

In [ ]:
star = mrex.Star(
    temperature=3500,
    radius=0.468,
    mass=0.495,
    phoenix_path="",
)

planet = mrex.Planet(
    radius=2.461,
    mass=7.2,
)

atm_h2he = mrex.Atmosphere(
    temperature=250,
    base_pressure=10e5,   # Pa = 10 bar
    top_pressure=1e-3,    # Pa = 1e-8 bar
    fill_gas=["H2", "He"],
    cia=["H2-H2", "H2-He"],
)

planet.set_atmosphere(atm_h2he)

system_h2 = mrex.System(
    planet=planet,
    star=star,
    sma=0.1429,
)

# CIA data are stored locally and reused on later runs.
mrex.Util.get_CIAs(["H2-H2", "H2-He"], path="CIA")

system_h2.make_tm()

In [ ]:
# Load the PandExo wavelength grid (um), then convert it to wavenumbers (cm^-1).
waves_um = np.loadtxt("waves.txt").astype(float)

# Keep the full NIRSpec/Prism wavelength grid used by the downstream workflow.
n_points = len(waves_um)
indices = np.linspace(0, n_points - 1, n_points, endpoint=True)
indices = np.round(indices).astype(int)

waves_um = waves_um[indices]
wn_grid = np.sort(10000.0 / waves_um)

In [ ]:
# Example trace gas configuration (single run before building the full grid).
system_h2.planet.atmosphere.add_gas("CO2", -2)
system_h2.planet.atmosphere.add_gas("CH4", -1)
system_h2.planet.atmosphere.add_gas("H2O", -3)
system_h2.planet.atmosphere.add_gas("NH3", -5)

system_h2.make_tm()
system_h2.plot_contributions(wn_grid)

## Representative spectrum

The next cells generate one spectrum before the full grid is built. This makes
it easier to inspect the transmission model and the contribution of the main
absorbers on the same wavelength grid used by the dataset.

## Contribution plot

The contribution plot separates molecular absorption, Rayleigh terms, and CIA
terms for the representative K2-18b-like spectrum. The H2-H2 CIA curve is used
as a visual mask so the trace-gas features remain readable.

In [ ]:
from taurex.binning import FluxBinner

tm = system_h2.transmission
nus = wn_grid  # cm^-1

tm_full_contrib=(tm.model_full_contrib(nus))
bn = FluxBinner(nus)

In [ ]:
import matplotlib as mpl
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

mpl.rcParams["font.family"] = "serif"
mpl.rcParams["mathtext.fontset"] = "stix"
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["font.size"] = 12

palette = {
    "dark_blue": "#264653",
    "teal": "#2A9D8F",
    "sandy": "#F4A261",
    "yellow": "#E9C46A",
    "burnt_red": "#E76F51",
    "grey_alpha": "#cccccc80",
}

target_gases = ["CH4", "CO2", "H2O", "NH3"]
gas_colors = {
    "CO2": palette["dark_blue"],
    "H2O": palette["teal"],
    "CH4": palette["yellow"],
    "NH3": palette["burnt_red"],
}

mixing_exp = {"CO2": -2, "CH4": -1, "H2O": -3, "NH3": -5}
gas_tex = {"CO2": r"$CO_2$", "H2O": r"$H_2O$", "CH4": r"$CH_4$", "NH3": r"$NH_3$"}
fill_color = palette["sandy"]


def exp_to_str(exp: float) -> str:
    """Format a log10 abundance as a compact decimal string."""
    return f"{10.0 ** exp:.2e}"


def _get_group(contrib, key: str):
    """Return a contribution group using a case-insensitive key match."""
    for k in contrib[1].keys():
        if str(k).strip().lower() == str(key).strip().lower():
            return contrib[1][k]
    return None


def _find_item_by_label(group, label: str):
    """Find a labelled item inside a list-style contribution group."""
    if group is None:
        return None
    for item in group:
        if str(item[0]).strip() == str(label).strip():
            return item
    return None


def get_array(contrib, mechanism: str, label: str):
    """Extract one contribution array from dict-style or list-style groups."""
    group = _get_group(contrib, mechanism)
    if group is None:
        return None

    if hasattr(group, "keys"):
        return np.array(group[label], float) if label in group else None

    item = _find_item_by_label(group, label)
    return np.array(item[1], float) if item is not None else None


def get_template_meta(contrib):
    """Reuse TauREx metadata from the first available contribution component."""
    abs_group = _get_group(contrib, "Absorption")
    if abs_group is not None and not hasattr(abs_group, "keys") and len(abs_group) > 0:
        item = abs_group[0]
        return item[2], item[3]

    cia_group = _get_group(contrib, "CIA")
    if cia_group is not None and not hasattr(cia_group, "keys") and len(cia_group) > 0:
        item = cia_group[0]
        return item[2], item[3]

    raise ValueError("Could not infer contribution metadata from Absorption or CIA groups.")


def iter_cia_terms(contrib):
    """Yield CIA contribution terms as ``(name, array)`` pairs."""
    cia_group = _get_group(contrib, "CIA")
    out = []

    if cia_group is None:
        return out

    if hasattr(cia_group, "keys"):
        for name, arr in cia_group.items():
            out.append((str(name), np.array(arr, float)))
        return out

    for item in cia_group:
        out.append((str(item[0]), np.array(item[1], float)))
    return out

In [ ]:
meta2, meta3 = get_template_meta(tm_full_contrib)

nus_tot, rp_rs_tot, *_ = bn.bin_model(tm.model(nus))
wl_tot = 1e4 / nus_tot
tot_ppm = 1e6 * rp_rs_tot

species_abs = []
species_ray = []
mins = [np.nanmin(tot_ppm)]

for mol in target_gases:
    abs_arr = get_array(tm_full_contrib, "Absorption", mol)
    ray_arr = get_array(tm_full_contrib, "Rayleigh", mol)

    if abs_arr is not None:
        obj = [tm_full_contrib[0], abs_arr, meta2, meta3]
        nus_m, rp_rs_m, *_ = bn.bin_model(obj)
        species_abs.append((mol, 1e4 / nus_m, 1e6 * rp_rs_m))
        mins.append(np.nanmin(1e6 * rp_rs_m))

    if ray_arr is not None:
        obj = [tm_full_contrib[0], ray_arr, meta2, meta3]
        nus_r, rp_rs_r, *_ = bn.bin_model(obj)
        species_ray.append((mol, 1e4 / nus_r, 1e6 * rp_rs_r))
        mins.append(np.nanmin(1e6 * rp_rs_r))

fill_curves = []
h2h2_curve = None

for name, arr in iter_cia_terms(tm_full_contrib):
    obj = [tm_full_contrib[0], arr, meta2, meta3]
    nus_c, rp_rs_c, *_ = bn.bin_model(obj)

    wl_c = 1e4 / nus_c
    y_c = 1e6 * rp_rs_c
    fill_curves.append((wl_c, y_c))
    mins.append(np.nanmin(y_c))

    if "H2-H2" in name:
        h2h2_curve = (wl_c, y_c)

for fg in ["H2", "He"]:
    arr = get_array(tm_full_contrib, "Rayleigh", fg)
    if arr is None:
        continue

    obj = [tm_full_contrib[0], arr, meta2, meta3]
    nus_r, rp_rs_r, *_ = bn.bin_model(obj)
    fill_curves.append((1e4 / nus_r, 1e6 * rp_rs_r))
    mins.append(np.nanmin(fill_curves[-1][1]))

ymin_global = float(np.nanmin(mins))

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))

for wl_f, y_f in fill_curves:
    ax.plot(wl_f, y_f, color=fill_color, lw=2.4, alpha=0.22, zorder=1, label="_nolegend_")

for mol, wl_r, y_r in species_ray:
    color = gas_colors.get(mol, palette["dark_blue"])
    ax.plot(wl_r, y_r, color=color, lw=2.2, alpha=0.22, zorder=1, label="_nolegend_")

for mol, wl_m, y_m in species_abs:
    color = gas_colors.get(mol, palette["dark_blue"])
    line, = ax.plot(wl_m, y_m, color=color, lw=2.5, alpha=0.8, zorder=2)
    line.set_path_effects([pe.Stroke(linewidth=2.8, foreground="#00000018"), pe.Normal()])

if h2h2_curve is not None:
    wl_h2, y_h2 = h2h2_curve
    ax.fill_between(wl_h2, ymin_global, y_h2, color="white", alpha=1.0, zorder=10)
    ax.plot(wl_h2, y_h2, color=fill_color, lw=2, alpha=0.5, zorder=10.1)

ax.fill_between(wl_tot, ymin_global, tot_ppm, color=palette["grey_alpha"], alpha=0.20, zorder=0)

tot_line, = ax.plot(wl_tot, tot_ppm, color=palette["grey_alpha"], lw=2, ls="--", zorder=11)
tot_line.set_path_effects(
    [
        pe.SimpleLineShadow(offset=(1.2, -1.2), alpha=0.25),
        pe.Stroke(linewidth=5.6, foreground="#00000022"),
        pe.Normal(),
    ]
)

handles = [Line2D([0], [0], color=palette["grey_alpha"], lw=1.0, ls="--")]
labels = ["Total spectrum"]

for gas in ["CH4", "CO2", "H2O", "NH3"]:
    handles.append(Line2D([0], [0], color=gas_colors[gas], lw=3.4, ls="-", alpha=0.8))
    labels.append(f"{gas_tex[gas]}: {exp_to_str(mixing_exp[gas])}")

ax.legend(
    handles=handles,
    labels=labels,
    loc="best",
    frameon=True,
    fancybox=False,
    framealpha=1.0,
    borderpad=0.8,
    labelspacing=0.55,
    handlelength=2.2,
    handletextpad=0.8,
    fontsize="small",
)

ax.set_xlabel("Wavelength [um]")
ax.set_ylabel(r"$(R_p/R_s)^2$ [ppm]")
ax.set_title("K2-18b analogue", fontsize=16)

plt.tight_layout()
# plt.savefig("K2-18b_spectrum_masked_top.png", dpi=500)
plt.show()

## Full spectral dataset

The grid below samples atmospheric temperature and trace-gas abundances, then
stores the generated spectra in `specs/K2-18b_data.joblib` for the training
notebook.

In [ ]:
parameter_space = {
    "planet.atmosphere.composition.NH3": {"min": -7, "max": -2, "n": 6, "include_absence": True},
    "planet.atmosphere.temperature": [250, 300, 350, 400, 450],
    "planet.atmosphere.composition.CH4": {"min": -8, "max": -1, "n": 8, "include_absence": True},
    "planet.atmosphere.composition.CO2": {"min": -10, "max": -1, "n": 10},
    "planet.atmosphere.composition.H2O": {"min": -7, "max": -2, "n": 6, "include_absence": True},
}

h2_data = system_h2.explore_parameter_space(
    wn_grid=wn_grid,
    parameter_space=parameter_space,
    header=True,
    observations=False,
    n_jobs=-1,
)

h2_data[["atm CO2", "atm H2O", "atm NH3", "atm CH4"]]

In [ ]:
import joblib

joblib.dump(h2_data, "specs/K2-18b_data.joblib")

spectra = h2_data.data
headers = h2_data.params

## Quick inspection

A small random subset is plotted to verify that the stored spectra and headers
are aligned.

In [ ]:
import random

n_samples = 5
random_indices = random.sample(range(len(spectra)), n_samples)

wavelengths_um = [float(col) for col in spectra.columns]

plt.figure(figsize=(12, 8))

for idx in random_indices:
    spec_values = spectra.iloc[idx].values
    params = headers.iloc[idx]

    temp_k = params["atm temperature"]
    base_p_bar = params["atm base_pressure"] / 1e5  # Pa -> bar

    co2 = params["atm CO2"]
    ch4 = params["atm CH4"]
    h2o = params["atm H2O"]
    nh3 = params["atm NH3"]

    plt.plot(
        wavelengths_um,
        spec_values,
        label=(
            f"T={temp_k} K, P={base_p_bar:.1f} bar, "
            f"CO2={co2}, CH4={ch4}, H2O={h2o}, NH3={nh3}"
        ),
    )

plt.xlabel("Wavelength [μm]")
plt.ylabel(r"$(R_p/R_s)^2$")
plt.title("Synthetic transmission spectra (random sample)")
plt.legend(loc="best", fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()